To run this, select an **A100 40GB** runtime and press *Runtime → Run all*.

This is a direct copy of the working Qwen GRPO notebook, adapted in place for the DeepSeek R1 Distill Qwen 1.5B SFT adapter.
legal extraction. The unrelated OpenMath tutorial block has been removed.

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import importlib.util, os
%pip install --upgrade -q uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pillow = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pillow = "numpy", "pillow"
    !uv pip install -q "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pillow} torchvision bitsandbytes "xformers==0.0.32.post2" "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -q unsloth
!uv pip install -q --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" "unsloth>=2026.4.2" unsloth_zoo
!uv pip install -q "transformers>=5.5.0,<5.6" "trl==0.22.2" "peft==0.20.0" "huggingface_hub>=1.5.0" "datasets==4.3.0" wandb weave pyarrow safetensors accelerate sentencepiece protobuf timm packaging
!uv pip install -q --no-build-isolation flash-linear-attention "causal_conv1d==1.6.0"
!uv pip install -q --no-deps "torchcodec==0.7.0" "apache-tvm-ffi==0.1.9" "tilelang==0.1.8" "torchao>=0.16.0"

In [ ]:
# Unsloth GRPO requires the compatible TRL 0.22.2 API.
# Import Weave now so TRL's W&B callback is ready.
import weave
# DeepSeek R1 Distill Qwen 1.5B is a text-only Qwen2 causal LM
# and uses Unsloth native inference.

### Unsloth

Goal: continue the existing **DeepSeek R1 Distill Qwen 1.5B SFT LoRA** with GRPO using the
Hugging Face Putusan `grpo` train/val splits, prioritizing substantive legal sections.
The fixed A100 profile matches the Qwen GRPO run: 24,576 cached prompt tokens, 4,096 cached completion tokens, and two generations within a 28,672-token total context. The existing Qwen GRPO prepared artifact is reused byte-for-byte without prompt rendering or tokenization. Step timing is measured during real training.

In [ ]:
from unsloth import FastLanguageModel

import gc, hashlib, json, math, os, random, re, shutil, tempfile, threading, time
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import wandb
from datasets import Dataset

SESSION_STARTED = time.perf_counter()
IN_COLAB = Path("/content").is_dir()
WORK_ROOT = (Path("/content/deepseek-r1-distill-qwen-1-5b-grpo-core") if IN_COLAB
             else Path.cwd() / "outputs/grpo/deepseek-r1-distill-qwen-1-5b-core")
if not torch.cuda.is_available():
    raise RuntimeError("Select one A100 40GB runtime.")
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 2**30
if "A100" not in gpu.name or not 38 <= gpu_gib <= 42:
    raise RuntimeError(f"Expected A100 40GB, found {gpu.name} ({gpu_gib:.2f} GiB).")

BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
EXPECTED_ARCHITECTURE = "Qwen2ForCausalLM"
INPUT_MODALITIES = ("text",)
WANDB_ENTITY = "haeriz42069-universitas-muhammadiyah-malang"
WANDB_PROJECT = "Sinergi-training"
WANDB_RUN_ID = "deepseek_r1_distill_qwen_1_5b_grpo_core_v1"
SFT_ADAPTER_ARTIFACT = f"{WANDB_ENTITY}/{WANDB_PROJECT}/deepseek-r1-distill-qwen-1-5b-section-sliced-lora:v0"
CHECKPOINT_ARTIFACT_NAME = "deepseek-r1-distill-qwen-1-5b-grpo-core-checkpoint"
FINAL_ADAPTER_ARTIFACT_NAME = "deepseek-r1-distill-qwen-1-5b-grpo-core-lora"
# Reuse the verified Qwen GRPO prompts and cached token counts without re-rendering.
PREPARED_DATASET_ARTIFACT_NAME = "qwen3-5-4b-grpo-core-prepared"
PREPARED_DATASET_ARTIFACT_VERSION = "v0"
EXPECTED_PREPARED_ARTIFACT_DIGEST = "78641aee01c9b4afdaaaa5b7edfb220e"
PREPARED_ROWS_REUSED_WITHOUT_RETOKENIZATION = True
PREPARED_DATASET_ARTIFACT = (
    f"{WANDB_ENTITY}/{WANDB_PROJECT}/{PREPARED_DATASET_ARTIFACT_NAME}:"
    f"{PREPARED_DATASET_ARTIFACT_VERSION}"
)
DATASET_IDENTITY = "Haeryz/putusan-structured-extraction"
DATASET_CONFIG = "grpo"
OUTPUT_ROOT = WORK_ROOT
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
FINAL_ADAPTER_DIR = OUTPUT_ROOT / "lora"
SOURCE_ADAPTER_ROOT = OUTPUT_ROOT / "source-sft-adapter"
for path in (CHECKPOINT_DIR, FINAL_ADAPTER_DIR, SOURCE_ADAPTER_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CORE_SECTIONS = (
    "dakwaan", "tuntutan", "saksi", "terdakwa", "fakta_hukum",
    "pertimbangan_hukum", "amar_putusan", "petunjuk_barang_bukti", "surat",
)
PROFILES = {
    "fixed28k": (24_576, 4_096),
}
TRAIN_EPOCHS = 1.0
GRADIENT_ACCUMULATION_STEPS = 1
SAVE_STEPS = 10
SESSION_LIMIT_SECONDS = 6 * 60 * 60
TRAINING_BUDGET_SECONDS = 5 * 60 * 60 + 15 * 60
SAFETY_FACTOR = 1.25
SEED = 3407
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

try:
    from google.colab import userdata
except ImportError:
    userdata = None
def secret(name):
    if value := os.getenv(name): return value
    if userdata is not None:
        try: return userdata.get(name)
        except Exception: return None
    return None
if not (wandb_key := secret("WANDB_API_KEY")):
    raise RuntimeError("Set WANDB_API_KEY in Colab Secrets or the environment.")
wandb.login(key=wandb_key, relogin=True)
if wandb.run is not None:
    wandb.finish(exit_code=1)
run = wandb.init(
    entity=WANDB_ENTITY, project=WANDB_PROJECT,
    name="deepseek-r1-distill-qwen-1-5b-grpo-core-legal", job_type="grpo",
    id=WANDB_RUN_ID, resume="allow",
    config={
        "base_model": BASE_MODEL,
        "source_sft_adapter": SFT_ADAPTER_ARTIFACT,
        "prepared_dataset_artifact": PREPARED_DATASET_ARTIFACT,
        "prepared_dataset_digest": EXPECTED_PREPARED_ARTIFACT_DIGEST,
        "prepared_rows_reused_without_retokenization": True,
    },
)
source_artifact = run.use_artifact(SFT_ADAPTER_ARTIFACT, type="model")
SOURCE_ADAPTER_DIGEST = source_artifact.digest
source_root = Path(source_artifact.download(
    root=str(SOURCE_ADAPTER_ROOT / SOURCE_ADAPTER_DIGEST)
))
run.config.update({"source_adapter_digest": SOURCE_ADAPTER_DIGEST}, allow_val_change=True)
adapter_config_files = sorted(source_root.rglob("adapter_config.json"))
if len(adapter_config_files) != 1:
    raise FileNotFoundError(
        f"Expected exactly one adapter_config.json in {source_root}, found {len(adapter_config_files)}"
    )
SOURCE_ADAPTER_DIR = adapter_config_files[0].parent
adapter_config = json.loads(adapter_config_files[0].read_text())
RECORDED_ADAPTER_BASE = str(adapter_config.get("base_model_name_or_path") or BASE_MODEL)
normalized_adapter_base = re.sub(r"[^a-z0-9]+", "", RECORDED_ADAPTER_BASE.lower())
if "deepseekr1distillqwen15b" not in normalized_adapter_base:
    raise RuntimeError(
        f"The pinned SFT adapter records an incompatible base model: {RECORDED_ADAPTER_BASE!r}"
    )
print(f"Pinned SFT adapter directory: {SOURCE_ADAPTER_DIR}")
print(f"Pinned SFT adapter base recorded as: {RECORDED_ADAPTER_BASE}")
run.config.update({"recorded_adapter_base": RECORDED_ADAPTER_BASE}, allow_val_change=True)

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(SOURCE_ADAPTER_DIR)
print(f"Work directory={WORK_ROOT}; GPU={gpu.name} ({gpu_gib:.2f} GiB)")

### Data Prep — full document, one prioritized legal section per GRPO unit

In [ ]:
PREPARED_DATASET_ROOT = OUTPUT_ROOT / "prepared-dataset"

prepared_artifact = run.use_artifact(
    PREPARED_DATASET_ARTIFACT,
    type="tokenized-dataset",
)
prepared_dir = Path(
    prepared_artifact.download(root=str(PREPARED_DATASET_ROOT))
)
print(f"W&B run: {run.url} (id={run.id}, resumed={run.resumed})")
if (EXPECTED_PREPARED_ARTIFACT_DIGEST is not None and
        prepared_artifact.digest != EXPECTED_PREPARED_ARTIFACT_DIGEST):
    raise RuntimeError("Prepared W&B artifact digest does not match the pinned artifact")

manifest_path = prepared_dir / "manifest.json"
train_path = prepared_dir / "train.parquet"
validation_path = prepared_dir / "validation.parquet"
for required_file in (manifest_path, train_path, validation_path):
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Prepared W&B artifact is missing {required_file.name}"
        )

prepared_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if prepared_manifest.get("preparation_version") != 1:
    raise RuntimeError("Unsupported prepared GRPO artifact version")
if prepared_manifest.get("dataset") != DATASET_IDENTITY:
    raise RuntimeError("Prepared artifact has the wrong source dataset")
if prepared_manifest.get("dataset_config") != DATASET_CONFIG:
    raise RuntimeError("Prepared artifact has the wrong dataset config")
if tuple(prepared_manifest.get("core_sections", ())) != CORE_SECTIONS:
    raise RuntimeError("Prepared row cache has different legal sections")

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

prepared_paths = {
    "train": train_path,
    "validation": validation_path,
}
for split_name, split_path in prepared_paths.items():
    expected_hash = prepared_manifest["splits"][split_name]["prepared_sha256"]
    actual_hash = file_sha256(split_path)
    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Prepared {split_name} SHA-256 mismatch: {actual_hash} != {expected_hash}"
        )

required_columns = {
    "id", "parent_id", "corpus", "section", "prompt", "answer",
    "input_text", "prompt_tokens", "answer_tokens", "sequence_tokens",
    "is_empty", "source_row_no",
}
prepared_frames = {
    split_name: pd.read_parquet(split_path)
    for split_name, split_path in prepared_paths.items()
}
for split_name, frame in prepared_frames.items():
    missing = required_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"Prepared {split_name} is missing columns: {sorted(missing)}"
        )
    expected_rows = prepared_manifest["splits"][split_name]["prepared_rows"]
    if len(frame) != expected_rows:
        raise RuntimeError(
            f"Prepared {split_name} row count {len(frame)} != {expected_rows}"
        )
    if frame["id"].duplicated().any():
        raise RuntimeError(f"Prepared {split_name} contains duplicate IDs")
    if (frame[["prompt_tokens", "answer_tokens", "sequence_tokens"]] <= 0).any().any():
        raise RuntimeError(f"Prepared {split_name} contains invalid token counts")
    if not (
        frame["sequence_tokens"]
        == frame["prompt_tokens"] + frame["answer_tokens"]
    ).all():
        raise RuntimeError(f"Prepared {split_name} token totals are inconsistent")

train_rows = prepared_frames["train"].to_dict("records")
validation_rows = prepared_frames["validation"].to_dict("records")
DATASET_FINGERPRINTS = {
    split_name: prepared_manifest["splits"][split_name]["prepared_sha256"]
    for split_name in ("train", "validation")
}
PREPARED_ARTIFACT_DIGEST = prepared_artifact.digest
run.config.update({
    "prepared_dataset_digest": PREPARED_ARTIFACT_DIGEST,
    "prepared_rows_reused_without_retokenization": True,
}, allow_val_change=True)
print(
    f"Loaded prepared W&B artifact {prepared_artifact.qualified_name}: "
    f"train={len(train_rows):,}, validation={len(validation_rows):,}; "
    "reused cached prompts and token counts without rendering or tokenization."
)

Let's look at the first row:

In [ ]:
train_rows[0]["prompt"][:2000]

In [ ]:
train_rows[0]["answer"]

In GSM8K, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [ ]:
def eligible(rows, profile_name):
    prompt_cap, completion_cap = PROFILES[profile_name]
    return [r for r in rows if r["prompt_tokens"] <= prompt_cap and r["answer_tokens"] <= completion_cap]

Let's map the dataset! and see the first row:

In [ ]:
coverage = []
for name, (prompt_cap, completion_cap) in PROFILES.items():
    train_fit, val_fit = eligible(train_rows, name), eligible(validation_rows, name)
    coverage.append({
        "profile": name, "max_prompt_length": prompt_cap,
        "max_completion_length": completion_cap, "max_seq_length": prompt_cap + completion_cap,
        "train_kept": len(train_fit), "train_coverage": len(train_fit) / len(train_rows),
        "validation_kept": len(val_fit), "validation_coverage": len(val_fit) / len(validation_rows),
    })
coverage_frame = pd.DataFrame(coverage)
display(coverage_frame)

We create a regex format to match the reasoning sections and answers:

In [ ]:
def completion_text(value):
    if isinstance(value, str): return value.strip()
    if isinstance(value, dict): return str(value.get("content", "")).strip()
    if isinstance(value, list) and value: return completion_text(value[-1])
    return str(value).strip()
def parse_prediction(value):
    try: parsed = json.loads(completion_text(value))
    except (json.JSONDecodeError, TypeError): return None
    return parsed if isinstance(parsed, dict) else None
def spans_for(parsed, section):
    if parsed is None or set(parsed) != {"sections", "empty_sections"}: return None
    sections = parsed.get("sections")
    if not isinstance(sections, dict) or set(sections) != {section}: return None
    spans = sections[section]
    if not isinstance(spans, list) or not all(isinstance(x, str) for x in spans): return None
    if parsed["empty_sections"] != ([section] if not spans else []): return None
    return spans

We verify it works:

In [ ]:
assert spans_for(json.loads(train_rows[0]["answer"]), train_rows[0]["section"]) is not None
assert parse_prediction("not json") is None

In [ ]:
# Reward helpers validated above.

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [ ]:
def reward_schema(completions, section, **kwargs):
    return [1.0 if spans_for(parse_prediction(c), s) is not None else 0.0
            for c, s in zip(completions, section, strict=True)]

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [ ]:
def reward_verbatim(completions, section, input_text, **kwargs):
    rewards = []
    for completion, requested, source in zip(completions, section, input_text, strict=True):
        spans = spans_for(parse_prediction(completion), requested)
        rewards.append(2.0 if spans is not None and all(span in source for span in spans) else 0.0)
    return rewards

Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [ ]:
def span_counter(spans): return Counter(token for span in spans for token in span.split())
def span_f1(predicted, gold):
    if not predicted and not gold: return 1.0
    p, g = span_counter(predicted), span_counter(gold)
    overlap = sum((p & g).values())
    if not overlap: return 0.0
    precision, recall = overlap / sum(p.values()), overlap / sum(g.values())
    return 2 * precision * recall / (precision + recall)
def reward_span_f1(completions, section, answer, **kwargs):
    rewards = []
    for completion, requested, gold_text in zip(completions, section, answer, strict=True):
        predicted = spans_for(parse_prediction(completion), requested)
        gold = json.loads(gold_text)["sections"][requested]
        rewards.append(0.0 if predicted is None else 5.0 * span_f1(predicted, gold))
    return rewards

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.

We also remove possible commas for example as in 123,456

In [ ]:
def reward_exact(completions, answer, **kwargs):
    return [2.0 if parse_prediction(c) == json.loads(g) else 0.0
            for c, g in zip(completions, answer, strict=True)]
REWARD_FUNCTIONS = [reward_schema, reward_verbatim, reward_span_f1, reward_exact]

We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [ ]:
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer
CHECKPOINT_METADATA_FILE = "sinergi_checkpoint.json"

def artifact_checkpoint_step(artifact):
    metadata = dict(artifact.metadata or {})
    try:
        step = int(metadata.get("global_step", 0))
    except (TypeError, ValueError):
        step = 0
    if step > 0:
        return step
    for alias in artifact.aliases or []:
        if match := re.fullmatch(r"step-(\d+)", str(alias)):
            return int(match.group(1))
    return None

def load_sft_adapter(max_seq_length):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(SOURCE_ADAPTER_DIR), max_seq_length=max_seq_length,
        dtype=torch.bfloat16, load_in_4bit=True, fast_inference=False,
        text_only=True, use_gradient_checkpointing="unsloth",
    )
    if not hasattr(model, "peft_config"):
        raise RuntimeError("SFT LoRA was not attached; refusing to add a new adapter.")
    architectures = tuple(getattr(model.config, "architectures", ()) or ())
    if architectures and EXPECTED_ARCHITECTURE not in architectures:
        raise RuntimeError(f"Unexpected DeepSeek architecture: {architectures}")
    config_dict = model.config.to_dict()
    if "vision_config" in config_dict or "audio_config" in config_dict:
        raise RuntimeError("DeepSeek R1 Distill Qwen 1.5B must remain text-only")
    return model, tokenizer



### Select the longest context profile that fits the measured A100 budget

The number of training steps is calculated from dataset size, batch size, and epochs.

In [ ]:
SELECTED_PROFILE = "fixed28k"
NUM_GENERATIONS = 2
MAX_PROMPT_LENGTH, MAX_COMPLETION_LENGTH = PROFILES[SELECTED_PROFILE]
MAX_SEQ_LENGTH = MAX_PROMPT_LENGTH + MAX_COMPLETION_LENGTH
run.config.update(
    {
        "context_profile": SELECTED_PROFILE,
        "max_prompt_length": MAX_PROMPT_LENGTH,
        "max_completion_length": MAX_COMPLETION_LENGTH,
        "num_generations": NUM_GENERATIONS,
    },
    allow_val_change=True,
)
eligible_train_rows = eligible(train_rows, SELECTED_PROFILE)
eligible_validation_rows = eligible(validation_rows, SELECTED_PROFILE)
CONTEXT_ELIGIBLE_TRAIN_ROWS = len(eligible_train_rows)
TRAIN_BATCH_SIZE = NUM_GENERATIONS
EVAL_BATCH_SIZE = NUM_GENERATIONS
WORLD_SIZE = torch.cuda.device_count()
GLOBAL_TRAIN_BATCH_SIZE = (
    TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * WORLD_SIZE
)
GLOBAL_EVAL_BATCH_SIZE = EVAL_BATCH_SIZE * WORLD_SIZE
if GLOBAL_TRAIN_BATCH_SIZE % NUM_GENERATIONS != 0:
    raise RuntimeError("Global training batch must be divisible by num_generations")
if GLOBAL_EVAL_BATCH_SIZE % NUM_GENERATIONS != 0:
    raise RuntimeError("Global evaluation batch must be divisible by num_generations")

train_dataset = Dataset.from_list(eligible_train_rows)
validation_dataset = Dataset.from_list(eligible_validation_rows)
PROMPTS_PER_OPTIMIZER_STEP = GLOBAL_TRAIN_BATCH_SIZE // NUM_GENERATIONS
if PROMPTS_PER_OPTIMIZER_STEP < 1:
    raise RuntimeError("GRPO must process at least one unique prompt per step")
STEPS_PER_EPOCH = math.ceil(
    len(train_dataset) / PROMPTS_PER_OPTIMIZER_STEP
)
TOTAL_TRAIN_STEPS = math.ceil(TRAIN_EPOCHS * STEPS_PER_EPOCH)
WARMUP_STEPS = max(1, math.ceil(TOTAL_TRAIN_STEPS * 0.10))

# Fixed, deterministic evaluation set: one median-length validation example
# per core legal section, with the three corpora assigned round-robin.
evaluation_rows = []
evaluation_corpora = sorted({row["corpus"] for row in eligible_validation_rows})
for section_index, section in enumerate(CORE_SECTIONS):
    corpus = evaluation_corpora[section_index % len(evaluation_corpora)]
    group = sorted(
        (row for row in eligible_validation_rows
         if row["corpus"] == corpus and row["section"] == section),
        key=lambda row: (row["sequence_tokens"], row["id"]),
    )
    if not group:
        raise RuntimeError(f"No validation row for corpus={corpus}, section={section}")
    evaluation_rows.append(group[len(group) // 2])
eval_padding = (-len(evaluation_rows)) % GLOBAL_EVAL_BATCH_SIZE
selected_eval_ids = {row["id"] for row in evaluation_rows}
extra_eval_candidates = sorted(
    (row for row in eligible_validation_rows if row["id"] not in selected_eval_ids),
    key=lambda row: (row["sequence_tokens"], row["id"]),
)
for _ in range(eval_padding):
    evaluation_rows.append(extra_eval_candidates.pop(len(extra_eval_candidates) // 2))
eval_dataset = Dataset.from_list(evaluation_rows)
if len(eval_dataset) % GLOBAL_EVAL_BATCH_SIZE != 0:
    raise RuntimeError("Evaluation rows do not form complete GRPO batches")
print(
    f"profile={SELECTED_PROFILE}; max_seq_length={MAX_SEQ_LENGTH}; "
    f"generations={NUM_GENERATIONS}; epochs={TRAIN_EPOCHS}; "
    f"steps_per_epoch={STEPS_PER_EPOCH}; total_steps={TOTAL_TRAIN_STEPS}; "
    f"train_raw={len(train_rows)}; train_eligible={len(train_dataset)}; "
    f"train_dropped_for_context={len(train_rows) - CONTEXT_ELIGIBLE_TRAIN_ROWS}; "
    f"validation_full={len(validation_dataset)}; "
    f"evaluation={len(eval_dataset)} (padding={eval_padding}) once after training; "
    f"warmup_steps={WARMUP_STEPS}"
)

<a name="Train"></a>
### Train the model

The validation panel is evaluated exactly once after training finishes and logged to W&B.

In [ ]:
identity = (
    f"core_legal_v4:{SELECTED_PROFILE}:g{NUM_GENERATIONS}:"
    f"b{TRAIN_BATCH_SIZE}:ga{GRADIENT_ACCUMULATION_STEPS}:e{TRAIN_EPOCHS:g}"
)
checkpoint_metadata = {
    "base_model": BASE_MODEL, "source_sft_adapter": SFT_ADAPTER_ARTIFACT,
    "source_adapter_digest": SOURCE_ADAPTER_DIGEST,
    "dataset": DATASET_IDENTITY, "dataset_config": DATASET_CONFIG,
    "dataset_fingerprints": DATASET_FINGERPRINTS, "train_curriculum": identity,
    "prepared_dataset_artifact": PREPARED_DATASET_ARTIFACT,
    "prepared_dataset_digest": PREPARED_ARTIFACT_DIGEST,
    "prepared_rows_reused_without_retokenization": True,
    "context_profile": SELECTED_PROFILE, "num_generations": NUM_GENERATIONS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "train_epochs": TRAIN_EPOCHS, "total_train_steps": TOTAL_TRAIN_STEPS,
    "conservative_seconds_per_step": None,
    "timed_training_steps": 0,
    "core_sections": list(CORE_SECTIONS),
    "eval_after_training": True,
    "eval_examples": len(eval_dataset),
    "warmup_steps": WARMUP_STEPS,
}

def checkpoint_matches(metadata):
    keys = ("base_model", "source_sft_adapter", "source_adapter_digest", "dataset",
            "dataset_config", "dataset_fingerprints", "train_curriculum",
            "prepared_dataset_artifact", "prepared_dataset_digest",
            "prepared_rows_reused_without_retokenization")
    return all(metadata.get(key) == checkpoint_metadata[key] for key in keys)

def newest_local_checkpoint():
    candidates = []
    for path in CHECKPOINT_DIR.glob("checkpoint-*"):
        try:
            step = int(path.name.removeprefix("checkpoint-"))
            metadata = json.loads((path / CHECKPOINT_METADATA_FILE).read_text())
        except (ValueError, OSError, json.JSONDecodeError):
            continue
        if (path / "trainer_state.json").is_file() and checkpoint_matches(metadata):
            candidates.append((step, path))
    return max(candidates, default=(0, None), key=lambda item: item[0])

def newest_remote_checkpoint():
    collection_name = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{CHECKPOINT_ARTIFACT_NAME}"
    api = wandb.Api()
    if not api.artifact_collection_exists(collection_name, "model-checkpoint"):
        return 0, None
    collection = api.artifact_collection("model-checkpoint", collection_name)
    candidates = []
    for artifact in collection.artifacts():
        step = artifact_checkpoint_step(artifact)
        if step and checkpoint_matches(dict(artifact.metadata or {})):
            candidates.append((step, artifact))
    return max(candidates, default=(0, None), key=lambda item: item[0])

def restore_newest_checkpoint():
    local_step, local_path = newest_local_checkpoint()
    remote_step, artifact = newest_remote_checkpoint()
    if artifact is None or local_step >= remote_step:
        return local_path
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    destination = CHECKPOINT_DIR / f"checkpoint-{remote_step}"
    if destination.exists():
        quarantine = CHECKPOINT_DIR / f".incomplete-{destination.name}-{int(time.time())}"
        os.replace(destination, quarantine)
    with tempfile.TemporaryDirectory(prefix=".wandb-restore-", dir=CHECKPOINT_DIR) as temporary:
        downloaded = Path(artifact.download(root=temporary)) / "checkpoint"
        state_file = downloaded / "trainer_state.json"
        if not state_file.is_file():
            raise RuntimeError("W&B checkpoint is missing checkpoint/trainer_state.json")
        saved_step = int(json.loads(state_file.read_text())["global_step"])
        if saved_step != remote_step:
            raise RuntimeError(f"W&B checkpoint step mismatch: {saved_step} != {remote_step}")
        os.replace(downloaded, destination)
    print(f"Restored W&B checkpoint step {remote_step}: {destination}")
    return destination

class WandbCheckpointUpload(TrainerCallback):
    @staticmethod
    def wait_until_committed(alias, timeout=3600):
        reference = (
            f"{WANDB_ENTITY}/{WANDB_PROJECT}/{CHECKPOINT_ARTIFACT_NAME}:"
            f"{alias}"
        )
        deadline = time.monotonic() + timeout
        while time.monotonic() < deadline:
            try:
                committed = wandb.Api().artifact(reference, type="model-checkpoint")
                if committed.state == "COMMITTED":
                    return committed
            except Exception:
                pass
            time.sleep(5)
        raise TimeoutError(f"W&B did not commit {reference} within {timeout}s")

    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        (checkpoint_dir / CHECKPOINT_METADATA_FILE).write_text(
            json.dumps(checkpoint_metadata, ensure_ascii=False, indent=2) + "\n"
        )
        artifact = wandb.Artifact(
            CHECKPOINT_ARTIFACT_NAME, type="model-checkpoint",
            metadata={**checkpoint_metadata, "global_step": int(state.global_step)},
        )
        artifact.add_dir(str(checkpoint_dir), name="checkpoint")
        step_alias = f"step-{state.global_step}"
        run.log_artifact(artifact, aliases=["latest", step_alias])
        committed = self.wait_until_committed(step_alias)
        print(f"W&B checkpoint committed: {committed.name}", flush=True)
        return control

def save_adapter(model, tokenizer, destination):
    destination.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(destination)
    tokenizer.save_pretrained(destination)

def log_final_adapter(metadata):
    artifact = wandb.Artifact(
        FINAL_ADAPTER_ARTIFACT_NAME, type="model", metadata=metadata
    )
    artifact.add_dir(str(FINAL_ADAPTER_DIR), name="adapter")
    run.log_artifact(artifact, aliases=["latest"])
    reference = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{FINAL_ADAPTER_ARTIFACT_NAME}:latest"
    deadline = time.monotonic() + 3600
    while time.monotonic() < deadline:
        try:
            committed = wandb.Api().artifact(reference, type="model")
            if committed.state == "COMMITTED": return committed
        except Exception:
            pass
        time.sleep(5)
    raise TimeoutError(f"W&B did not commit {reference} within 3600s")

restored_checkpoint = restore_newest_checkpoint()

class RuntimeDeadline(TrainerCallback):
    def __init__(self):
        self.last = None
        self.intervals = []
        self.initial_estimate = 600.0

    def on_train_begin(self, args, state, control, **kwargs):
        self.last = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        now = time.perf_counter()
        if self.last is not None:
            self.intervals.append(now - self.last)
            self.intervals = self.intervals[-5:]
        self.last = now
        measured = max(self.intervals, default=self.initial_estimate)
        conservative = measured * SAFETY_FACTOR
        checkpoint_metadata["conservative_seconds_per_step"] = conservative
        checkpoint_metadata["timed_training_steps"] = len(self.intervals)
        run.summary["runtime/conservative_seconds_per_step"] = conservative
        run.summary["runtime/session_step_capacity"] = int(
            TRAINING_BUDGET_SECONDS // conservative
        )
        reserve = max(1200.0, 2 * conservative + 300.0)
        if now - SESSION_STARTED + reserve + conservative >= SESSION_LIMIT_SECONDS:
            control.should_save = True
            control.should_evaluate = False
            control.should_training_stop = True
            print(
                "Deadline guard: saving and committing a W&B checkpoint before stopping.",
                flush=True,
            )
        return control

for stale_name in ("trainer", "model", "tokenizer"):
    globals().pop(stale_name, None)
gc.collect(); torch.cuda.empty_cache()
model, tokenizer = load_sft_adapter(MAX_SEQ_LENGTH)
training_args = GRPOConfig(
    output_dir=str(CHECKPOINT_DIR),
    use_vllm=False,
    bf16=True,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    unsloth_grpo_mini_batch=TRAIN_BATCH_SIZE,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_train_epochs=TRAIN_EPOCHS,
    # max_steps=100,  # Uncomment to stop early; positive max_steps overrides num_train_epochs.
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    do_eval=True,
    eval_strategy="no",
    report_to="wandb",
    remove_unused_columns=False,
    mask_truncated_completions=True,
    temperature=0.6,
    top_p=0.95,
    seed=SEED,
)
if training_args.unsloth_grpo_mini_batch < 1:
    raise RuntimeError("Unsloth GRPO mini-batch divisor must be positive")
configured_train_global_batch = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
    * WORLD_SIZE
)
configured_eval_global_batch = training_args.per_device_eval_batch_size * WORLD_SIZE
if configured_train_global_batch % training_args.num_generations != 0:
    raise RuntimeError("Configured global training batch is not GRPO-divisible")
if configured_eval_global_batch % training_args.num_generations != 0:
    raise RuntimeError("Configured global evaluation batch is not GRPO-divisible")

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:
trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, reward_funcs=REWARD_FUNCTIONS,
    args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset,
)
trainer.add_callback(WandbCheckpointUpload())
trainer.add_callback(RuntimeDeadline())
heartbeat_stop = threading.Event()
heartbeat_started = time.perf_counter()
def training_heartbeat():
    while not heartbeat_stop.wait(60):
        elapsed = time.perf_counter() - heartbeat_started
        step = int(trainer.state.global_step)
        print(
            f"Training alive: step={step}/{TOTAL_TRAIN_STEPS}; "
            f"training elapsed={elapsed / 60:.1f} min; W&B={run.url}",
            flush=True,
        )
        run.summary["runtime/heartbeat_step"] = step
        run.summary["runtime/heartbeat_elapsed_seconds"] = elapsed
heartbeat_thread = threading.Thread(target=training_heartbeat, daemon=True)
heartbeat_thread.start()
try:
    trainer_stats = trainer.train(
        resume_from_checkpoint=str(restored_checkpoint) if restored_checkpoint else None
    )
finally:
    heartbeat_stop.set()
    heartbeat_thread.join(timeout=5)
print(trainer_stats)
print(f"Running the single final validation at step {trainer.state.global_step}...", flush=True)
final_eval_metrics = trainer.evaluate(eval_dataset=eval_dataset, metric_key_prefix="eval")
trainer.save_metrics("eval", final_eval_metrics)
run.summary["eval/final_global_step"] = int(trainer.state.global_step)
print(final_eval_metrics)

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
# Optional smoke inference; disabled so Run All spends the protected budget on training.
if False:
    FastLanguageModel.for_inference(model)
    row = validation_dataset[0]
    inputs = tokenizer(row["prompt"], return_tensors="pt", add_special_tokens=False).to("cuda")
    output = model.generate(**inputs, max_new_tokens=min(4096, MAX_COMPLETION_LENGTH), do_sample=False)
    print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
save_adapter(model, tokenizer, FINAL_ADAPTER_DIR)
final_metadata = {**checkpoint_metadata, **dict(trainer_stats.metrics),
                  "final_global_step": int(trainer.state.global_step)}
final_artifact = log_final_adapter(final_metadata)
print(f"Final GRPO LoRA committed: {final_artifact.name}")
run.finish(exit_code=0)

Verify LoRA is actually trained!

In [ ]:
required = {"adapter_config.json"}
missing = required - {p.name for p in FINAL_ADAPTER_DIR.iterdir()}
if missing: raise RuntimeError(f"Final adapter is incomplete: {sorted(missing)}")
print(f"Verified local adapter: {FINAL_ADAPTER_DIR}")

Now we load the LoRA and test:

In [ ]:
print(f"Resume collection: {WANDB_ENTITY}/{WANDB_PROJECT}/{CHECKPOINT_ARTIFACT_NAME}:latest")
print("Every standard checkpoint was synchronously committed before training continued.")

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("deepseek_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/deepseek_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("deepseek_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/deepseek_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("deepseek_lora")
    tokenizer.save_pretrained("deepseek_lora")
if False:
    model.push_to_hub("HF_USERNAME/deepseek_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/deepseek_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("deepseek_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/deepseek_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("deepseek_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/deepseek_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("deepseek_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/deepseek_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/deepseek_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>